# Gold Layer — SARIMAX Forecasting & Anomaly Detection

Trains SARIMAX per city × target, evaluates on held-out test set, retrains on full data, produces 30-day forecast, and flags anomalies (95th percentile).

**SARIMAX config:** order=(2,1,1), seasonal_order=(1,0,1,7), annual Fourier exogenous regressors (2 harmonics)

| Column | Type | Description |
|--------|------|-------------|
| `DATE` | `datetime64` | Date |
| `CITY` | `str` | State or union territory name |
| `TEMPERATURE_C` | `float64` | Actual or forecasted |
| `PRECIPITATION_MM` | `float64` | Actual or forecasted |
| `WIND_SPEED_KMH` | `float64` | Actual or forecasted |
| `SOURCE` | `str` | `historical` or `forecast` |
| `ANOMALY_TEMPERATURE` | `bool` | Exceeds 95th percentile |
| `ANOMALY_PRECIPITATION` | `bool` | Exceeds 95th percentile |
| `ANOMALY_WIND` | `bool` | Exceeds 95th percentile |

In [ ]:
df = spark.read.table("silver_weather_india.weather.processed_weather")
df = df.toPandas()
df.columns = [c.upper() for c in df.columns]
df["DATE"] = pd.to_datetime(df["DATE"])

print(f"Silver: {len(df):,} rows | {df['CITY'].nunique()} regions | "
      f"{df['DATE'].min().date()} -> {df['DATE'].max().date()}")

## Imports & Hyperparameters

In [ ]:
import warnings
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore")

TARGETS    = ["TEMPERATURE_C", "PRECIPITATION_MM", "WIND_SPEED_KMH"]
TEST_DAYS  = 30
FORECAST_DAYS = 30
ANOMALY_PERCENTILE = 95

# SARIMAX configuration
ORDER          = (2, 1, 1)       # (p, d, q)
SEASONAL_ORDER = (1, 0, 1, 7)    # (P, D, Q, s) — weekly seasonality

TARGET_LABELS = {
    "TEMPERATURE_C":    "Temperature (C)",
    "PRECIPITATION_MM": "Precipitation (mm)",
    "WIND_SPEED_KMH":   "Wind Speed (km/h)",
}

cities = sorted(df["CITY"].unique())
n_cities = len(cities)
print(f"Regions: {n_cities}")
print(f"SARIMAX order={ORDER}, seasonal_order={SEASONAL_ORDER}")

# This notebook fits SARIMAX per city/target directly off the silver table
# rather than pivoting into a DATE x CITY array, but it relies on the same
# invariant: every date must have a row for every region, or a city's series
# will silently have implicit gaps that desynchronize it from the calendar.
per_date = df.groupby("DATE")["CITY"].nunique()
incomplete = per_date[per_date < n_cities]
if len(incomplete) > 0 or df[TARGETS].isna().any().any():
    raise ValueError(
        f"Silver data has holes: {len(incomplete)} date(s) are missing rows for "
        f"one or more of the {n_cities} regions, and/or target columns contain "
        "NaN. This means the DATE x CITY grid is incomplete. Re-run the silver "
        "layer, which drops dates that are not present for every region."
    )

## Helper Functions

In [ ]:
def build_fourier_exog(dates_series):
    """Build annual Fourier exogenous regressors (2 harmonics) from dates."""
    doy = dates_series.dt.dayofyear.values.astype(float)
    return np.column_stack([
        np.sin(2 * np.pi * doy / 365.25),
        np.cos(2 * np.pi * doy / 365.25),
        np.sin(4 * np.pi * doy / 365.25),
        np.cos(4 * np.pi * doy / 365.25),
    ])


def fit_sarimax(y, exog):
    """Fit a SARIMAX model and return the fitted result."""
    model = SARIMAX(y, exog=exog, order=ORDER, seasonal_order=SEASONAL_ORDER,
                    enforce_stationarity=False, enforce_invertibility=False)
    return model.fit(disp=False, maxiter=200)


def forecast_sarimax(result, n_steps, exog_future, target):
    """Produce n_steps forecast from a fitted SARIMAX result."""
    preds = result.forecast(steps=n_steps, exog=exog_future)
    if target == "PRECIPITATION_MM":
        preds = np.maximum(0.0, preds)
    return preds

## Phase 1: Evaluation — Train on data[:-30], Test on Last 30 Days

In [ ]:
last_date   = df["DATE"].max()
test_cutoff = last_date - pd.Timedelta(days=TEST_DAYS)
train_df    = df[df["DATE"] <= test_cutoff].copy()
test_df     = df[df["DATE"] > test_cutoff].copy()

print(f"Train: {len(train_df):,} rows | Test: {len(test_df):,} rows")
print(f"Test period: {test_df['DATE'].min().date()} -> {test_df['DATE'].max().date()}")

mae_per_target = np.zeros(len(TARGETS))
n_total = n_cities * len(TARGETS)
done = 0

for city in cities:
    city_train = train_df[train_df["CITY"] == city].sort_values("DATE").reset_index(drop=True)
    city_test  = test_df[test_df["CITY"] == city].sort_values("DATE").reset_index(drop=True)
    exog_train = build_fourier_exog(city_train["DATE"])

    # Exog for test period
    test_dates = pd.date_range(start=city_train["DATE"].max() + pd.Timedelta(days=1),
                               periods=TEST_DAYS, freq="D")
    exog_test = build_fourier_exog(pd.Series(test_dates))

    for ti, target in enumerate(TARGETS):
        done += 1
        y = city_train[target].values.astype(float)
        result = fit_sarimax(y, exog_train)
        preds  = forecast_sarimax(result, TEST_DAYS, exog_test, target)

        actuals = city_test[target].values[:TEST_DAYS]
        mae = mean_absolute_error(actuals, preds[:len(actuals)])
        mae_per_target[ti] += mae

    print(f"  {city} done ({done}/{n_total})")

mae_per_target /= n_cities
print(f"\nTest MAE (avg across {n_cities} regions):")
for ti, t in enumerate(TARGETS):
    print(f"  {TARGET_LABELS[t]:<22} {mae_per_target[ti]:.3f}")

## Phase 2: Full Retrain & 30-Day Forecast

In [ ]:
last_date = df["DATE"].max()
forecast_dates = pd.date_range(start=last_date + pd.Timedelta(days=1),
                               periods=FORECAST_DAYS, freq="D")
exog_future = build_fourier_exog(pd.Series(forecast_dates))

print(f"Retraining on full data and forecasting {FORECAST_DAYS} days...")
print(f"Forecast period: {forecast_dates[0].date()} -> {forecast_dates[-1].date()}")

forecast_rows = []
done = 0

for city in cities:
    city_df = df[df["CITY"] == city].sort_values("DATE").reset_index(drop=True)
    exog_full = build_fourier_exog(city_df["DATE"])

    city_preds = {}
    for target in TARGETS:
        done += 1
        y = city_df[target].values.astype(float)
        result = fit_sarimax(y, exog_full)
        preds  = forecast_sarimax(result, FORECAST_DAYS, exog_future, target)
        city_preds[target] = preds

    for step in range(FORECAST_DAYS):
        forecast_rows.append({
            "DATE": forecast_dates[step],
            "CITY": city,
            "TEMPERATURE_C":    float(city_preds["TEMPERATURE_C"][step]),
            "PRECIPITATION_MM": float(city_preds["PRECIPITATION_MM"][step]),
            "WIND_SPEED_KMH":   float(city_preds["WIND_SPEED_KMH"][step]),
            "SOURCE": "forecast",
        })
    print(f"  {city} done ({done}/{n_total})")

forecast_df = pd.DataFrame(forecast_rows)
print(f"\nForecast: {len(forecast_df)} rows ({forecast_dates[0].date()} -> {forecast_dates[-1].date()})")

## Anomaly Detection & Assemble Gold DataFrame

In [ ]:
# 95th percentile thresholds (combined across all cities)
hist_df = df.copy()
thresholds = {t: hist_df[t].quantile(ANOMALY_PERCENTILE / 100) for t in TARGETS}

print(f"{'Variable':<25} {'Threshold':>10}")
print("-" * 37)
for t in TARGETS:
    print(f"{TARGET_LABELS[t]:<25} {thresholds[t]:>10.2f}")

# Flag anomalies
forecast_df["ANOMALY_TEMPERATURE"]   = forecast_df["TEMPERATURE_C"]    > thresholds["TEMPERATURE_C"]
forecast_df["ANOMALY_PRECIPITATION"] = forecast_df["PRECIPITATION_MM"] > thresholds["PRECIPITATION_MM"]
forecast_df["ANOMALY_WIND"]          = forecast_df["WIND_SPEED_KMH"]   > thresholds["WIND_SPEED_KMH"]

# Assemble: historical + forecast
hist_gold = hist_df.copy()
hist_gold["SOURCE"]                = "historical"
hist_gold["ANOMALY_TEMPERATURE"]   = False
hist_gold["ANOMALY_PRECIPITATION"] = False
hist_gold["ANOMALY_WIND"]          = False

gold_df = pd.concat([hist_gold, forecast_df], ignore_index=True)
gold_df = gold_df.sort_values(["CITY", "DATE"]).reset_index(drop=True)

n_hist = (gold_df["SOURCE"] == "historical").sum()
n_fore = (gold_df["SOURCE"] == "forecast").sum()
n_anom = (gold_df["ANOMALY_TEMPERATURE"] | gold_df["ANOMALY_PRECIPITATION"] | gold_df["ANOMALY_WIND"]).sum()
print(f"\ngold_df: {len(gold_df):,} rows ({n_hist:,} historical + {n_fore} forecast, {n_anom} anomalies)")

# Build anomaly summary for LLM recommendations
def build_anomaly_summary(row):
    flags = []
    if row["ANOMALY_TEMPERATURE"]:
        flags.append("extreme heat")
    if row["ANOMALY_PRECIPITATION"]:
        flags.append("heavy precipitation")
    if row["ANOMALY_WIND"]:
        flags.append("strong winds")
    return ", ".join(flags) if flags else None

gold_df["combined_anomaly"] = gold_df.apply(build_anomaly_summary, axis=1)

# Ensure column types are Spark-compatible (avoid CANNOT_DETERMINE_TYPE on all-None columns)
gold_df["DATE"]                 = pd.to_datetime(gold_df["DATE"])
gold_df["TEMPERATURE_C"]        = gold_df["TEMPERATURE_C"].astype(float)
gold_df["PRECIPITATION_MM"]     = gold_df["PRECIPITATION_MM"].astype(float)
gold_df["WIND_SPEED_KMH"]       = gold_df["WIND_SPEED_KMH"].astype(float)
gold_df["ANOMALY_TEMPERATURE"]  = gold_df["ANOMALY_TEMPERATURE"].astype(bool)
gold_df["ANOMALY_PRECIPITATION"] = gold_df["ANOMALY_PRECIPITATION"].astype(bool)
gold_df["ANOMALY_WIND"]         = gold_df["ANOMALY_WIND"].astype(bool)
# combined_anomaly is string or None — cast to object so Spark gets explicit StringType
gold_df["combined_anomaly"]     = gold_df["combined_anomaly"].astype(object)

# Convert to Spark DataFrame with explicit schema to avoid type inference issues
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, BooleanType, TimestampType
)

spark = SparkSession.builder.getOrCreate()

spark_schema = StructType([
    StructField("DATE",                 TimestampType(), True),
    StructField("CITY",                 StringType(),    True),
    StructField("TEMPERATURE_C",        DoubleType(),    True),
    StructField("PRECIPITATION_MM",     DoubleType(),    True),
    StructField("WIND_SPEED_KMH",       DoubleType(),    True),
    StructField("SOURCE",               StringType(),    True),
    StructField("ANOMALY_TEMPERATURE",  BooleanType(),   True),
    StructField("ANOMALY_PRECIPITATION",BooleanType(),   True),
    StructField("ANOMALY_WIND",         BooleanType(),   True),
    StructField("combined_anomaly",     StringType(),    True),
])

gold_sdf = spark.createDataFrame(gold_df[list(f.name for f in spark_schema.fields)], schema=spark_schema)

# Generate LLM recommendations for anomalous days
from pyspark.sql.functions import expr, when, col

gold_sdf = gold_sdf.withColumn(
    "RECOMMENDATION",
    when(
        col("combined_anomaly").isNotNull(),
        expr("""query_model(
            'default.oci_ai_models.google.gemini-2.5-pro',
            CONCAT('Give exactly 3 brief safety tips as a numbered list. No introduction, no headers, no markdown formatting. Plain text only. For residents facing:', combined_anomaly),
            map('max_tokens', '1024')
        )""")
    ).otherwise(None)
)
gold_sdf.show()

In [ ]:
gold_sdf.filter(col("RECOMMENDATION").isNotNull()).select("RECOMMENDATION").show(1, truncate=False)

In [ ]:
# Create gold catalog
gold_catalog = "gold_weather_india"
gold_schema = "weather"
gold_table = "forecasted_weather"

gold_catalog_adb = "gold_weather_india_adb"
gold_schema_adb = "weather"
gold_table_adb = "FORECASTED_WEATHER"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {gold_catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_catalog}.{gold_schema}")

# Register as temp view first
gold_sdf.createOrReplaceTempView("gold_temp")

# Use SQL DDL to create Delta table
spark.sql(f"""
  CREATE OR REPLACE TABLE {gold_catalog}.{gold_schema}.{gold_table}
  USING DELTA
  AS SELECT * FROM gold_temp
""")

spark.sql(f"""
  CREATE OR REPLACE TABLE {gold_catalog_adb}.{gold_schema_adb}.{gold_table_adb}
  USING DELTA
  AS SELECT * FROM gold_temp
""")

In [ ]:
df = spark.read.table("gold_weather_india.weather.forecasted_weather")
df.show()